# Comparison Data and Simulation

- from Madeleine Ginolin
- adapted by SDC
- **date** : 2026-10-26
- **kernel on my laptop**: `conda_py313`
- **kernel on NERSC :** `desc-td-env` (Not -dev)
  
The adaptation wrt the original is on the downloading of DUST maps

To copy files at NERSC:
```bash
scp final_sample_ddf.parquet dagoret@dtn01.nersc.gov:/pscratch/sd/d/dagoret/SimTtranscientLSSTSurvey/sprints/Example_notebooks_madeleine2026/.
```

In [ ]:
import skysurvey
import numpy as np
import os
import pandas
import sncosmo
from scipy.stats import norm, lognorm
import scipy
import astropy
from astropy import cosmology
from scipy.special import expit
import time
from scipy.interpolate import CubicSpline
import matplotlib.pyplot as plt
from shapely import geometry
from astropy.coordinates import SkyCoord
import astropy.units as u
from nested_pandas import read_parquet

# Simulate the SNe

Note: This is Mi's file, should update once we get a better list of candidates (someone said on the Slack that some candidates might be variable stars + Vincenzo cross-matched with known AGNs)

In [ ]:
true_sne = read_parquet("final_sample_ddf.parquet")

In [ ]:
logs_full = pandas.read_parquet("edp2_detectorvisit_table.parquet")
footprint_ccd = 9.6 / 189
side_ccd = np.sqrt(footprint_ccd)
footprint = geometry.Polygon(
    (
        (-side_ccd / 2, -side_ccd / 2),
        (-side_ccd / 2, side_ccd / 2),
        (side_ccd / 2, side_ccd / 2),
        (side_ccd / 2, -side_ccd / 2),
    )
)

In [ ]:
bands = ["g", "r", "i", "z", "u", "y"]
for band in bands:
    # Bandpasses from the butler
    filename = "filter_lsst_" + band + ".npy"
    bandpass = np.load(filename)
    # Here we trim the light curves to prevent a bug, but if/where we cut is TBD with the whole WG
    sncosmo.register(
        sncosmo.Bandpass(
            bandpass[0],
            bandpass[1],
            wave_unit=u.nm,
            name="rubin" + band,  # trim_level=0.001,
            normalize=True,
        )
    )
    logs_full.loc[logs_full["band"] == "lsst" + band, "band"] = "rubin" + band

In [ ]:
dust_effect = {
    "effect": sncosmo.models.F99Dust(r_v=3.1),
    "name": "mw",
    "frame": "obs",
    "model": {
        "mwebv": {
            "func": skysurvey.effects.milkyway.get_mwebv,
            "kwargs": {"ra": "@ra", "dec": "@dec", "which": "sfd"},
        }
    },
}

In [ ]:
sncosmo_model = sncosmo.Model(source=sncosmo.get_source("salt3"))
snia = skysurvey.SNeIa()
snia.set_template(sncosmo_model)
snia.set_cosmology(cosmology.Planck18)
snia.add_effect(dust_effect)

In [ ]:
# We need to define those as we have x0 from the fits


def x0_to_magobs(x0, sncosmo_model):
    return -2.5 * np.log10(x0) + sncosmo_model.bandmag("bessellb", "ab", 0)


def magobs_to_magabs(magobs, z, cosmo):
    return magobs - cosmo.distmod(np.asarray(z, dtype="float32")).value

In [ ]:
# Create the snia object from the data
snia_data = pandas.DataFrame(
    {
        "z": true_sne["z"],
        "x1": true_sne["x1"],
        "c": true_sne["c"],
        "t0": true_sne["t0"],
        "ra": true_sne["ra_dia_object_lc"],
        "dec": true_sne["dec_dia_object_lc"],
        "mwebv": true_sne["mwebv"],
        "x0": true_sne["x0"],
    }
)
snia_data["magobs"] = x0_to_magobs(snia_data["x0"], sncosmo_model)
snia_data["magabs"] = magobs_to_magabs(snia_data["magobs"], snia_data["z"], cosmology.Planck18)

In [ ]:
len(snia_data)

In [ ]:
snia_data

In [ ]:
%%time

lsst = skysurvey.Survey.from_pointings(logs_full, footprint=footprint)

snia_data["sn"] = ["LSST_" + str(k) for k in list(snia_data.index)]
snia.set_data(snia_data)

dset = skysurvey.DataSet.from_targets_and_survey(snia, lsst, incl_error=True, phase_range=[-260, 260])
index = list(dset.data.index.get_level_values(0))
lcs = dset.data.assign(sn=["LSST_" + str(ind) for ind in index], magsys="ab")

In [ ]:
lcs["flux_prime"] = lcs["flux"] * 10 ** (0.4 * (31.4 - lcs["zp"]))
lcs["fluxerr_prime"] = lcs["fluxerr"] * 10 ** (0.4 * (31.4 - lcs["zp"]))
lcs["zp_prime"] = 31.4  # ZP to match the one from the data (which is nJy)

In [ ]:
lcs

Note: We are skipping:
- Coadds, as the data is not-coadded
- Searcheff, because it's a probabilistic process so it wouldn't match the data, but we thus expect more points in the sims that in the data
- LC quality cuts, as all the SNe we simulate should have well sampled LCs since the data went through quality cuts already

In [ ]:
# We compuse the SNR of every point, which we could also use to compare data and sism (TO DO?)

nsigma_det = 5  # Level of detection considered
lcs["ndetection"] = lcs["flux"] / lcs["fluxerr"]

# Comparison of true vs simulated LCs 

In [ ]:
# Select a SN, either a random one or the one you want
i = np.random.randint(0, 145)
i = 0
sn_name = "LSST_" + str(i)

In [ ]:
sn = true_sne.iloc[i]
params = {"z": sn["z"], "t0": sn["t0"], "x0": sn["x0"], "x1": sn["x1"], "c": sn["c"], "mwebv": sn["mwebv"]}

# Getting the LC from the data
data_lc = pandas.DataFrame(sn["diaObjectForcedSource_dia_object_lc"])
data_lc.rename(
    columns={"midpointMjdTai": "mjd", "psfDiffFlux": "flux", "psfDiffFluxErr": "fluxerr"}, inplace=True
)
data_lc["newband"] = "rubin" + data_lc["band"]

In [ ]:
params

In [ ]:
data_lc

In [ ]:
sim_lc = lcs[lcs["sn"] == sn_name]

In [ ]:
sim_lc

In [ ]:
phase_min_restframe = -300
phase_max_restframe = 300

peaktime = params["t0"]
phase_min = params["t0"] + phase_min_restframe * (1 + params["z"])
phase_max = params["t0"] + phase_max_restframe * (1 + params["z"])

In [ ]:
bands = ["rubinu", "rubing", "rubinr", "rubini", "rubinz", "rubiny"]
c = ["C0", "C1", "C2", "C3", "C4", "C5"]

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(14, 6), sharex=True, sharey=True)

for i in range(3):
    for j in range(2):
        band = bands[2 * i + j]
        ax[j, i].set_title(band)
        data_lc_band = data_lc[(data_lc["newband"] == band) & (data_lc["mjd"].between(phase_min, phase_max))]
        sim_lc_band = sim_lc[(sim_lc["band"] == band) & (sim_lc["mjd"].between(phase_min, phase_max))]
        ax[j, i].scatter(
            data_lc_band["mjd"] - peaktime,
            data_lc_band["flux"],
            s=5,
            label="Data",
            c=c[2 * i + j],
            marker="<",
        )
        # ax[j,i].errorbar(data_lc_band['mjd']-peaktime, data_lc_band['flux'], yerr=data_lc_band['fluxerr'],
        #            fmt='none', zorder=2, alpha=0.5, c=c[2*i+j], label='Data')
        ax[j, i].scatter(
            sim_lc_band["mjd"] - peaktime,
            sim_lc_band["flux_prime"],
            s=5,
            label="Sims",
            c=c[2 * i + j],
            marker=">",
        )
        # eb1 = ax[j,i].errorbar(sim_lc_band['mjd']-peaktime, sim_lc_band['flux_prime'], yerr=sim_lc_band['fluxerr_prime'],
        #           fmt='none', zorder=2, alpha=0.8, c='grey', label='Sims')
        # eb1[-1][0].set_linestyle('dotted')
        ax[j, i].legend()
        ax[j, i].spines["top"].set_visible(False)
        ax[j, i].spines["right"].set_visible(False)
        ax[j, i].spines["left"].set_visible(False)
        ax[j, i].spines["bottom"].set_visible(False)

# ax.text(-40, 375, r'$z^\mathrm{true}=$'+str(round(sn['z'].iloc[0], 3))+r', $x_1^\mathrm{true}=$'+str(round(sn['x1'].iloc[0], 3))+r', $c^\mathrm{true}=$'+str(round(sn['c'].iloc[0], 3)))
# ax.text(-40, 400, r'$z^\mathrm{fit}=$'+str(round(sn_fit['z'].iloc[0], 3))+r'$\pm$'+str(round(sn_fit['z_err'].iloc[0], 3))+r', $x_1^\mathrm{fit}=$'+str(round(sn_fit['x1'].iloc[0], 3))+r'$\pm$'+str(round(sn_fit['x1_err'].iloc[0], 3))+r', $c^\mathrm{fit}=$'+str(round(sn_fit['c'].iloc[0], 3))+r'$\pm$'+str(round(sn_fit['c_err'].iloc[0], 3)))

# ax.set_yticks([0])
# ax.set_ylabel('Flux')
# ax.set_xlabel(r'Observer-frame phase (in days)')


# ax[0,0].set_xlim(-40, 60)
# ax[0,0].set_ylim(0, 5000)

plt.tight_layout()
# plt.savefig('/Users/mginolin/Documents/example_LC_elaiss_DP2.pdf', format='pdf', dpi=1200, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)

for i in range(len(bands)):
    band = bands[i]
    data_lc_band = data_lc[(data_lc["newband"] == band) & (data_lc["mjd"].between(phase_min, phase_max))]
    sim_lc_band = sim_lc[(sim_lc["band"] == band) & (sim_lc["mjd"].between(phase_min, phase_max))]
    ax[0].scatter(data_lc_band["mjd"] - peaktime, data_lc_band["flux"], s=5, label=band, c=c[i])
    ax[0].errorbar(
        data_lc_band["mjd"] - peaktime,
        data_lc_band["flux"],
        yerr=data_lc_band["fluxerr"],
        fmt="none",
        zorder=2,
        alpha=0.5,
        c=c[i],
    )
    ax[1].scatter(sim_lc_band["mjd"] - peaktime, sim_lc_band["flux_prime"], s=5, label=band, c=c[i])
    ax[1].errorbar(
        sim_lc_band["mjd"] - peaktime,
        sim_lc_band["flux_prime"],
        yerr=sim_lc_band["fluxerr_prime"],
        fmt="none",
        zorder=2,
        alpha=0.5,
        c=c[i],
    )

for i in range(2):
    ax[i].legend()
    ax[i].spines["top"].set_visible(False)
    ax[i].spines["right"].set_visible(False)
    ax[i].spines["left"].set_visible(False)
    ax[i].spines["bottom"].set_visible(False)
    ax[i].set_xlabel("Observer-frame phase (in days)")
    ax[i].set_ylabel("Flux (in nJy)")
    ax[i].set_xlim(-20, 40)
    ax[i].set_ylim(-2000, 8000)

ax[0].set_title("Data")
ax[1].set_title("Simulations")
# ax.text(-40, 375, r'$z^\mathrm{true}=$'+str(round(sn['z'].iloc[0], 3))+r', $x_1^\mathrm{true}=$'+str(round(sn['x1'].iloc[0], 3))+r', $c^\mathrm{true}=$'+str(round(sn['c'].iloc[0], 3)))
# ax.text(-40, 400, r'$z^\mathrm{fit}=$'+str(round(sn_fit['z'].iloc[0], 3))+r'$\pm$'+str(round(sn_fit['z_err'].iloc[0], 3))+r', $x_1^\mathrm{fit}=$'+str(round(sn_fit['x1'].iloc[0], 3))+r'$\pm$'+str(round(sn_fit['x1_err'].iloc[0], 3))+r', $c^\mathrm{fit}=$'+str(round(sn_fit['c'].iloc[0], 3))+r'$\pm$'+str(round(sn_fit['c_err'].iloc[0], 3)))

# ax.set_yticks([0])
# ax.set_ylabel('Flux')
# ax.set_xlabel(r'Observer-frame phase (in days)')


plt.tight_layout()
# plt.savefig('/Users/mginolin/Documents/example_LC_elaiss_DP2.pdf', format='pdf', dpi=1200, bbox_inches='tight')
plt.show()

# Full survey plots

In [ ]:
len(snia_data)

## Per band plots

In [ ]:
# Histogram of the flux errors (for all the data points at once, no per point matching)

In [ ]:
pulls = []

for i in range(6):
    band = bands[i]
    pull_band = []
    for i in range(len(snia_data)):
        sn_name = "LSST_" + str(i)
        sn = true_sne.iloc[i]
        # Data
        data_lc = pandas.DataFrame(sn["diaObjectForcedSource_dia_object_lc"])
        data_lc.rename(
            columns={"midpointMjdTai": "mjd", "psfDiffFlux": "flux", "psfDiffFluxErr": "fluxerr"},
            inplace=True,
        )
        data_lc["newband"] = "rubin" + data_lc["band"]
        data_lc_band = data_lc[(data_lc["newband"] == band) & (data_lc["mjd"].between(phase_min, phase_max))]
        # Sim
        sim_lc_band = lcs[
            (lcs["sn"] == sn_name) & (lcs["band"] == band) & (lcs["mjd"].between(phase_min, phase_max))
        ]
        # Merge
        lc_band_merged = pandas.merge_asof(
            sim_lc_band[["mjd", "flux_prime", "fluxerr_prime"]].astype("float32"),
            data_lc_band[["mjd", "flux", "fluxerr"]].astype("float32"),
            on="mjd",
            direction="nearest",
            tolerance=0.01,
            suffixes=("_sim", "_data"),
        )
        value = lc_band_merged["flux_prime"] - lc_band_merged["flux"]
        err = np.sqrt(lc_band_merged["fluxerr_prime"] ** 2 + lc_band_merged["fluxerr"] ** 2)
        pull_band += list(value / err)
    pulls.append(np.array(pull_band))

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(12, 4))

bins_pull = np.linspace(-6, 6, 40)
xpull = np.linspace(-6, 6, 1000)

for i in range(3):
    for j in range(2):
        band = bands[2 * i + j]
        ax[j, i].hist(pulls[2 * i + j], bins=bins_pull, color=c[2 * i + j], alpha=0.5, density=True)
        ax[j, i].plot(xpull, norm.pdf(xpull), label=r"$\mathcal{N}(0,1)$", color=c[2 * i + j], alpha=0.8)
        ax[j, i].legend()
        ax[j, i].set_title(band)
        ax[j, i].spines["top"].set_visible(False)
        ax[j, i].spines["right"].set_visible(False)
        ax[j, i].spines["left"].set_visible(False)
        ax[j, i].spines["bottom"].set_visible(False)

ax[1, 0].set_ylabel("PDF")
ax[0, 0].set_ylabel("PDF")

fig.suptitle(
    r"Pulls of sims vs data $\frac{F^{sim}-F^{data}}{\sqrt{(F_\mathrm{err}^{sim})^2+(F_\mathrm{err}^{data})^2}}$"
)

plt.tight_layout()
# plt.savefig('/Users/mginolin/Documents/DESC_plots/EDP2_pulls_per_band.pdf', format='pdf', dpi=1200, bbox_inches='tight')
plt.show()

In [ ]:
# Flux ratios for a point-to-point matching

In [ ]:
diff_fluxerr = []

for i in range(6):
    band = bands[i]
    diff_band = []
    for i in range(len(snia_data)):
        sn_name = "LSST_" + str(i)
        sn = true_sne.iloc[i]
        # Data
        data_lc = pandas.DataFrame(sn["diaObjectForcedSource_dia_object_lc"])
        data_lc.rename(
            columns={"midpointMjdTai": "mjd", "psfDiffFlux": "flux", "psfDiffFluxErr": "fluxerr"},
            inplace=True,
        )
        data_lc["newband"] = "rubin" + data_lc["band"]
        data_lc_band = data_lc[(data_lc["newband"] == band) & (data_lc["mjd"].between(phase_min, phase_max))]
        # Sim
        sim_lc_band = lcs[
            (lcs["sn"] == sn_name) & (lcs["band"] == band) & (lcs["mjd"].between(phase_min, phase_max))
        ]
        # Merge
        lc_band_merged = pandas.merge_asof(
            sim_lc_band[["mjd", "flux_prime", "fluxerr_prime"]].astype("float32"),
            data_lc_band[["mjd", "flux", "fluxerr"]].astype("float32"),
            on="mjd",
            direction="nearest",
            tolerance=0.01,
            suffixes=("_sim", "_data"),
        )
        diff_band += list(lc_band_merged["fluxerr_prime"] / lc_band_merged["fluxerr"])
    diff_fluxerr.append(np.array(diff_band))

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(12, 4))

bins_diff = np.linspace(0, 2.1, 40)

for i in range(3):
    for j in range(2):
        band = bands[2 * i + j]
        ax[j, i].hist(diff_fluxerr[2 * i + j], bins=bins_diff, label=band, color=c[2 * i + j], alpha=0.5)
        ax[j, i].legend()
        ax[j, i].spines["top"].set_visible(False)
        ax[j, i].spines["right"].set_visible(False)
        ax[j, i].spines["left"].set_visible(False)
        ax[j, i].spines["bottom"].set_visible(False)

ax[1, 0].set_ylabel("Number of points")
ax[0, 0].set_ylabel("Number of points")

fig.suptitle(r"Ratio of the sims and data errorbars $\frac{F_\mathrm{err}^{sim}}{F_\mathrm{err}^{data}}$")

plt.tight_layout()
# plt.savefig('/Users/mginolin/Documents/DESC_plots/EDP2_fluxerr_ratios_per_band.pdf', format='pdf', dpi=1200, bbox_inches='tight')
plt.show()

In [ ]:
# Pulls of the fluxes (this depends on the quality of the SALT fits, and will be impacted by contamination in the sample)

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(12, 4))

bins_fluxerr = np.linspace(0, 3000, 40)

for i in range(3):
    for j in range(2):
        band = bands[2 * i + j]
        fluxerr_sim_band = lcs[lcs["band"] == band]["fluxerr_prime"]
        fluxerr_data_band = pandas.concat(
            [
                pandas.DataFrame(
                    true_sne.iloc[i]["diaObjectForcedSource_dia_object_lc"][
                        true_sne.iloc[i]["diaObjectForcedSource_dia_object_lc"]["band"] == band[-1]
                    ]
                )["psfDiffFluxErr"]
                for i in range(len(true_sne))
            ],
            ignore_index=True,
        )
        ax[j, i].hist(fluxerr_sim_band, bins=bins_fluxerr, label="Sims", color=c[2 * i + j], alpha=0.5)
        ax[j, i].hist(
            fluxerr_data_band,
            bins=bins_fluxerr,
            label="Data",
            color=c[2 * i + j],
            alpha=1,
            histtype="step",
            linewidth=2,
        )
        ax[j, i].legend()
        ax[j, i].set_title(band)
        ax[j, i].spines["top"].set_visible(False)
        ax[j, i].spines["right"].set_visible(False)
        ax[j, i].spines["left"].set_visible(False)
        ax[j, i].spines["bottom"].set_visible(False)

ax[1, 0].set_ylabel("Number of points")
ax[0, 0].set_ylabel("Number of points")

ax[1, 0].set_xlabel("Flux error")
ax[1, 1].set_xlabel("Flux error")
ax[1, 2].set_xlabel("Flux error")

fig.suptitle(r"Histogram of the sims and data errorbars")

plt.tight_layout()
# plt.savefig(
#    "/Users/mginolin/Documents/DESC_plots/EDP2_fluxerr_hist_per_band.pdf",
#    format="pdf",
#    dpi=1200,
#    bbox_inches="tight",
# )
plt.savefig(
    "EDP2_fluxerr_hist_per_band.pdf",
    format="pdf",
    dpi=1200,
    bbox_inches="tight",
)
plt.show()

## Plots per DDF

In [ ]:
plt.scatter(true_sne["ra_dia_object_lc"], true_sne["dec_dia_object_lc"], s=1)
plt.xlabel("Ra")
plt.ylabel("Dec")
plt.show()

In [ ]:
ra_ranges = [[0, 20], [40, 80], [40, 80], [100, 160]]
dec_ranges = [[-60, -35], [-60, -35], [-35, -10], [-10, 20]]
ddf_name = ["ELAISS1", "EDFS", "ECDFS", "COSMOS"]

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(8, 4))

bins_fluxerr = np.linspace(0, 3000, 40)

for i in range(2):
    for j in range(2):
        ra_range = ra_ranges[2 * i + j]
        dec_range = dec_ranges[2 * i + j]
        sn_ddf = snia_data[
            snia_data["ra"].between(ra_range[0], ra_range[1])
            & snia_data["dec"].between(dec_range[0], dec_range[1])
        ]
        indexes = list(sn_ddf.index.get_level_values(0))
        fluxerr_sim_band = lcs[lcs["sn"].isin(sn_ddf["sn"])]["fluxerr_prime"]
        fluxerr_data_band = pandas.concat(
            [
                pandas.DataFrame(true_sne.iloc[i]["diaObjectForcedSource_dia_object_lc"])["psfDiffFluxErr"]
                for i in indexes
            ],
            ignore_index=True,
        )
        ax[j, i].hist(fluxerr_sim_band, bins=bins_fluxerr, label="Sims", color="teal", alpha=0.5)
        ax[j, i].hist(
            fluxerr_data_band,
            bins=bins_fluxerr,
            label="Data",
            color="darkred",
            alpha=1,
            histtype="step",
            linewidth=2,
        )
        ax[j, i].legend()
        ax[j, i].set_title(ddf_name[2 * i + j])
        ax[j, i].spines["top"].set_visible(False)
        ax[j, i].spines["right"].set_visible(False)
        ax[j, i].spines["left"].set_visible(False)
        ax[j, i].spines["bottom"].set_visible(False)

ax[1, 0].set_ylabel("Number of points")
ax[0, 0].set_ylabel("Number of points")

ax[1, 0].set_xlabel("Flux error")
ax[1, 1].set_xlabel("Flux error")

fig.suptitle(r"Histogram of the sims and data errorbars")

plt.tight_layout()
# plt.savefig('/Users/mginolin/Documents/DESC_plots/EDP2_fluxerr_hist_per_DDF.pdf', format='pdf', dpi=1200, bbox_inches='tight')
plt.show()

In [ ]:
diff_fluxerr = []

for i in range(4):
    ra_range = ra_ranges[i]
    dec_range = dec_ranges[i]
    sn_ddf = snia_data[
        snia_data["ra"].between(ra_range[0], ra_range[1])
        & snia_data["dec"].between(dec_range[0], dec_range[1])
    ]
    indexes = list(sn_ddf.index.get_level_values(0))
    diff_ddf = []
    for i in indexes:
        sn_name = "LSST_" + str(i)
        sn = true_sne.iloc[i]
        # Data
        data_lc = pandas.DataFrame(sn["diaObjectForcedSource_dia_object_lc"])
        data_lc.rename(
            columns={"midpointMjdTai": "mjd", "psfDiffFlux": "flux", "psfDiffFluxErr": "fluxerr"},
            inplace=True,
        )
        data_lc_band = data_lc[data_lc["mjd"].between(phase_min, phase_max)]
        # Sim
        sim_lc_band = lcs[(lcs["sn"] == sn_name) & (lcs["mjd"].between(phase_min, phase_max))]
        # Merge
        lc_band_merged = pandas.merge_asof(
            sim_lc_band[["mjd", "flux_prime", "fluxerr_prime"]].astype("float32"),
            data_lc_band[["mjd", "flux", "fluxerr"]].astype("float32"),
            on="mjd",
            direction="nearest",
            tolerance=0.01,
            suffixes=("_sim", "_data"),
        )
        diff_ddf += list(lc_band_merged["fluxerr_prime"] / lc_band_merged["fluxerr"])
    diff_fluxerr.append(np.array(diff_ddf))

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(8, 4))

bins_diff = np.linspace(0, 2.1, 40)

for i in range(2):
    for j in range(2):
        ax[j, i].hist(diff_fluxerr[2 * i + j], bins=bins_diff, color="teal", alpha=0.5)
        ax[j, i].set_title(ddf_name[2 * i + j])
        ax[j, i].spines["top"].set_visible(False)
        ax[j, i].spines["right"].set_visible(False)
        ax[j, i].spines["left"].set_visible(False)
        ax[j, i].spines["bottom"].set_visible(False)

ax[1, 0].set_ylabel("Number of points")
ax[0, 0].set_ylabel("Number of points")

fig.suptitle(r"Ratio of the sims and data errorbars $\frac{F_\mathrm{err}^{sim}}{F_\mathrm{err}^{data}}$")

plt.tight_layout()
# plt.savefig('/Users/mginolin/Documents/DESC_plots/EDP2_fluxerr_ratios_per_DDF.pdf', format='pdf', dpi=1200, bbox_inches='tight')
plt.show()

In [ ]:
pulls = []

for i in range(4):
    ra_range = ra_ranges[i]
    dec_range = dec_ranges[i]
    sn_ddf = snia_data[
        snia_data["ra"].between(ra_range[0], ra_range[1])
        & snia_data["dec"].between(dec_range[0], dec_range[1])
    ]
    indexes = list(sn_ddf.index.get_level_values(0))
    pulls_ddf = []
    for i in indexes:
        sn_name = "LSST_" + str(i)
        sn = true_sne.iloc[i]
        # Data
        data_lc = pandas.DataFrame(sn["diaObjectForcedSource_dia_object_lc"])
        data_lc.rename(
            columns={"midpointMjdTai": "mjd", "psfDiffFlux": "flux", "psfDiffFluxErr": "fluxerr"},
            inplace=True,
        )
        data_lc_band = data_lc[data_lc["mjd"].between(phase_min, phase_max)]
        # Sim
        sim_lc_band = lcs[(lcs["sn"] == sn_name) & (lcs["mjd"].between(phase_min, phase_max))]
        # Merge
        lc_band_merged = pandas.merge_asof(
            sim_lc_band[["mjd", "flux_prime", "fluxerr_prime"]].astype("float32"),
            data_lc_band[["mjd", "flux", "fluxerr"]].astype("float32"),
            on="mjd",
            direction="nearest",
            tolerance=0.01,
            suffixes=("_sim", "_data"),
        )
        value = lc_band_merged["flux_prime"] - lc_band_merged["flux"]
        err = np.sqrt(lc_band_merged["fluxerr_prime"] ** 2 + lc_band_merged["fluxerr"] ** 2)
        pulls_ddf += list(value / err)
    pulls.append(np.array(pulls_ddf))

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(8, 4))

bins_pull = np.linspace(-6, 6, 40)
xpull = np.linspace(-6, 6, 1000)

for i in range(2):
    for j in range(2):
        ax[j, i].hist(pulls[2 * i + j], bins=bins_pull, color="teal", alpha=0.5, density=True)
        ax[j, i].plot(xpull, norm.pdf(xpull), color="darkred", label=r"$\mathcal{N}(0,1)$")
        ax[j, i].set_title(ddf_name[2 * i + j])
        ax[j, i].legend()
        ax[j, i].spines["top"].set_visible(False)
        ax[j, i].spines["right"].set_visible(False)
        ax[j, i].spines["left"].set_visible(False)
        ax[j, i].spines["bottom"].set_visible(False)

ax[1, 0].set_ylabel("PDF")
ax[0, 0].set_ylabel("PDF")

fig.suptitle(
    r"Pulls of sims vs data $\frac{F^{sim}-F^{data}}{\sqrt{(F_\mathrm{err}^{sim})^2+(F_\mathrm{err}^{data})^2}}$"
)

plt.tight_layout()
# plt.savefig('/Users/mginolin/Documents/DESC_plots/EDP2_pulls_per_DDF.pdf', format='pdf', dpi=1200, bbox_inches='tight')
plt.show()

In [ ]:
# Should probably trim the data to a given phase range

# SALT fits

Not sure if this is actually useful but can do pools (should look at Mi's notebook for reference)

In [ ]:
def fit_salt_integrated(lc, z_guess, t0_guess, mwebv_guess, fit_z=False):
    model = sncosmo.Model(
        source=sncosmo.get_source("salt3-nir"),
        effects=[sncosmo.F99Dust(r_v=3.1)],
        effect_names=["mw"],
        effect_frames=["obs"],
    )
    keymap = {}
    lc_dict = {
        key: lc[keymap.get(key, key)].values for key in ["time", "band", "flux", "fluxerr", "zp", "zpsys"]
    }
    if fit_z == False:
        model.set(z=z_guess, t0=t0_guess, mwebv=mwebv_guess)
        try:
            result, fitted_model = sncosmo.fit_lc(
                lc_dict,
                model,
                ["t0", "x0", "x1", "c"],  # parameters of model to vary
                bounds={"x1": (-10, 10), "c": (-1, 3)},
                modelcov=True,
            )  # bounds on parameters (if any)
        except (RuntimeError, np.linalg.LinAlgError):
            return np.ones(24) * np.nan
        if result["success"] == False:
            return np.ones(24) * np.nan
        return np.concatenate(
            [
                result["parameters"],
                result["covariance"].flatten(),
                np.array([result["chisq"], result["ndof"]]),
            ]
        )
    else:
        model.set(t0=t0_guess, mwebv=mwebv_guess)
        try:
            result, fitted_model = sncosmo.fit_lc(
                lc_dict,
                model,
                ["t0", "x0", "x1", "c", "z"],  # parameters of model to vary
                bounds={"x1": (-10, 10), "c": (-1, 3), "z": (0, 0.5)},
                modelcov=True,
            )  # bounds on parameters (if any)
        except (RuntimeError, np.linalg.LinAlgError):
            return np.ones(33) * np.nan
        if result["success"] == False:
            return np.ones(33) * np.nan
        return np.concatenate(
            [
                result["parameters"],
                result["covariance"].flatten(),
                np.array([result["chisq"], result["ndof"]]),
            ]
        )

### Inputs for the SALT fits

`lcs_coadd`, `snias_new`, `snias_detected` and `targets_to_consider` come from `Simulate_DDF.ipynb` (coadd, search efficiency and quality-cut steps). Those steps are skipped in this notebook, so the objects are rebuilt here directly from the simulated light curves `lcs` (no coadd, no search efficiency), with the same selection: at least 5$\sigma$ detections in at least 3 bands, within a phase range of [-10, 40] days.

In [ ]:
nsigma_det = 5  # Detection level considered

# Light curves in the format expected by fit_salt_integrated (no coadd here: one point per simulated visit)
lcs_coadd = pandas.DataFrame(
    {
        "sn": lcs["sn"].values,
        "band": lcs["band"].values,
        "time": lcs["mjd"].values,
        "flux": lcs["flux_prime"].values,
        "fluxerr": lcs["fluxerr_prime"].values,
        "zp": 31.4,  # same zero point as flux_prime (nJy)
        "zpsys": "ab",
    }
)
lcs_coadd["ndetection"] = lcs_coadd["flux"] / lcs_coadd["fluxerr"]

# Keep only valid points (finite flux, finite and strictly positive error): NaN/zero errors break the fit covariance
valid = np.isfinite(lcs_coadd["flux"]) & np.isfinite(lcs_coadd["fluxerr"]) & (lcs_coadd["fluxerr"] > 0)
print("Invalid points removed:", int((~valid).sum()))
lcs_coadd = lcs_coadd[valid].reset_index(drop=True)

snias_new = snia_data.copy()

# Rest-frame phase of every point
data_merged = lcs_coadd.merge(snias_new[["sn", "t0", "z"]], on="sn")
data_merged["phase"] = (data_merged["time"] - data_merged["t0"]) / (1 + data_merged["z"])

# Number of >= nsigma_det detections per SN and per band, in the phase range [-10, 40] days
ndet = (
    data_merged[(data_merged["ndetection"] >= nsigma_det) & data_merged["phase"].between(-10, 40)]
    .groupby(["sn", "band"])
    .size()
)
nbands = ndet.groupby(level=0).size()

# SNe detected in at least 3 bands
targets_to_consider = nbands[nbands >= 3].index
snias_detected = snias_new[snias_new["sn"].isin(targets_to_consider)]

# Restrict the light curves to the phase range where SALT3 is defined (-20 to +50 days). The simulation was run
# with phase_range=[-260, 260]; points far from peak can make the fit covariance singular (LinAlgError)
lcs_coadd = (
    data_merged[data_merged["phase"].between(-20, 50)]
    .drop(columns=["t0", "z", "phase"])
    .reset_index(drop=True)
)

print(len(targets_to_consider), "SNe selected out of", len(snias_new))

### SALT fits Without z info

In [ ]:
%%time

salt_fits = []

fit_z = True

for i in range(len(targets_to_consider)):
    sn_name = targets_to_consider[i]
    lc = lcs_coadd[lcs_coadd["sn"] == targets_to_consider[i]]
    sn = snias_new[snias_new["sn"] == targets_to_consider[i]]
    df_result = fit_salt_integrated(
        lc,
        z_guess=sn["z"].iloc[0],
        t0_guess=sn["t0"].iloc[0] + norm.rvs(0, 10),
        mwebv_guess=sn["mwebv"].iloc[0],
        fit_z=fit_z,
    )
    salt_fits.append(np.array(df_result))
salt_fits = np.array(salt_fits)

if fit_z == False:
    df_salt = pandas.DataFrame(
        salt_fits,
        columns=[
            "z",
            "t0",
            "x0",
            "x1",
            "c",
            "mwebv",
            "cov_t0_t0",
            "cov_t0_x0",
            "cov_t0_x1",
            "cov_t0_c",
            "cov_x0_t0",
            "cov_x0_x0",
            "cov_x0_x1",
            "cov_x0_c",
            "cov_x1_t0",
            "cov_x1_x0",
            "cov_x1_x1",
            "cov_x1_c",
            "cov_c_t0",
            "cov_c_x0",
            "cov_c_x1",
            "cov_c_c",
            "chisq",
            "ndof",
        ],
    )
else:
    df_salt = pandas.DataFrame(
        salt_fits,
        columns=[
            "z",
            "t0",
            "x0",
            "x1",
            "c",
            "mwebv",
            "cov_z_z",
            "cov_z_t0",
            "cov_z_x0",
            "cov_z_x1",
            "cov_z_c",
            "cov_t0_z",
            "cov_t0_t0",
            "cov_t0_x0",
            "cov_t0_x1",
            "cov_t0_c",
            "cov_x0_z",
            "cov_x0_t0",
            "cov_x0_x0",
            "cov_x0_x1",
            "cov_x0_c",
            "cov_x1_z",
            "cov_x1_t0",
            "cov_x1_x0",
            "cov_x1_x1",
            "cov_x1_c",
            "cov_c_z",
            "cov_c_t0",
            "cov_c_x0",
            "cov_c_x1",
            "cov_c_c",
            "chisq",
            "ndof",
        ],
    )
    df_salt["z_err"] = np.sqrt(df_salt["cov_z_z"])
df_salt["t0_err"] = np.sqrt(df_salt["cov_t0_t0"])
df_salt["x0_err"] = np.sqrt(df_salt["cov_x0_x0"])
df_salt["x1_err"] = np.sqrt(df_salt["cov_x1_x1"])
df_salt["c_err"] = np.sqrt(df_salt["cov_c_c"])
df_salt["sn"] = list(targets_to_consider)
df_salt["fitprob"] = scipy.stats.chi2.sf(df_salt["chisq"], df_salt["ndof"])

In [ ]:
mask_c = df_salt["c"].between(-0.3, 0.3) & (df_salt["c_err"] < 0.2)
mask_x1 = df_salt["x1"].between(-3, 3) & (df_salt["x1_err"] < 1)
mask_fit = (df_salt["t0_err"] < 2) & (df_salt["fitprob"] > 1e-4) & (df_salt["mwebv"] < 0.3)

df_salt_selected = df_salt[mask_c & mask_x1 & mask_fit]
snias_new_selected = snias_new[snias_new["sn"].isin(df_salt_selected["sn"])]

In [ ]:
df_salt_selected.sort_values(by="sn", inplace=True)
df_salt_selected.reset_index(inplace=True)
snias_new_selected.sort_values(by="sn", inplace=True)
snias_new_selected.reset_index(inplace=True)

In [ ]:
df_salt_selected["mb"] = -2.5 * np.log10(df_salt_selected["x0"]) + 10.502
df_salt_selected["mb_err"] = 2.5 * df_salt_selected["x0_err"] / (np.log(10) * df_salt_selected["x0"])
snias_new_selected["mb"] = -2.5 * np.log10(snias_new_selected["x0"]) + 10.502

In [ ]:
print("Sucessful SALT fit")
print(len(snias_detected[snias_detected["sn"].isin(df_salt.dropna()["sn"])]))

In [ ]:
print("After SALT cuts")
print(len(snias_new_selected))

### SALT fit with z info

In [ ]:
%%time

salt_fits = []

fit_z = False

for i in range(len(targets_to_consider)):
    sn_name = targets_to_consider[i]
    lc = lcs_coadd[lcs_coadd["sn"] == targets_to_consider[i]]
    sn = snias_new[snias_new["sn"] == targets_to_consider[i]]
    df_result = fit_salt_integrated(
        lc,
        z_guess=sn["z"].iloc[0],
        t0_guess=sn["t0"].iloc[0] + norm.rvs(0, 10),
        mwebv_guess=sn["mwebv"].iloc[0],
        fit_z=fit_z,
    )
    salt_fits.append(np.array(df_result))
salt_fits = np.array(salt_fits)

if fit_z == False:
    df_salt = pandas.DataFrame(
        salt_fits,
        columns=[
            "z",
            "t0",
            "x0",
            "x1",
            "c",
            "mwebv",
            "cov_t0_t0",
            "cov_t0_x0",
            "cov_t0_x1",
            "cov_t0_c",
            "cov_x0_t0",
            "cov_x0_x0",
            "cov_x0_x1",
            "cov_x0_c",
            "cov_x1_t0",
            "cov_x1_x0",
            "cov_x1_x1",
            "cov_x1_c",
            "cov_c_t0",
            "cov_c_x0",
            "cov_c_x1",
            "cov_c_c",
            "chisq",
            "ndof",
        ],
    )
else:
    df_salt = pandas.DataFrame(
        salt_fits,
        columns=[
            "z",
            "t0",
            "x0",
            "x1",
            "c",
            "mwebv",
            "cov_z_z",
            "cov_z_t0",
            "cov_z_x0",
            "cov_z_x1",
            "cov_z_c",
            "cov_t0_z",
            "cov_t0_t0",
            "cov_t0_x0",
            "cov_t0_x1",
            "cov_t0_c",
            "cov_x0_z",
            "cov_x0_t0",
            "cov_x0_x0",
            "cov_x0_x1",
            "cov_x0_c",
            "cov_x1_z",
            "cov_x1_t0",
            "cov_x1_x0",
            "cov_x1_x1",
            "cov_x1_c",
            "cov_c_z",
            "cov_c_t0",
            "cov_c_x0",
            "cov_c_x1",
            "cov_c_c",
            "chisq",
            "ndof",
        ],
    )
    df_salt["z_err"] = np.sqrt(df_salt["cov_z_z"])
df_salt["t0_err"] = np.sqrt(df_salt["cov_t0_t0"])
df_salt["x0_err"] = np.sqrt(df_salt["cov_x0_x0"])
df_salt["x1_err"] = np.sqrt(df_salt["cov_x1_x1"])
df_salt["c_err"] = np.sqrt(df_salt["cov_c_c"])
df_salt["sn"] = list(targets_to_consider)
df_salt["fitprob"] = scipy.stats.chi2.sf(df_salt["chisq"], df_salt["ndof"])

In [ ]:
mask_c = df_salt["c"].between(-0.3, 0.3) & (df_salt["c_err"] < 0.2)
mask_x1 = df_salt["x1"].between(-3, 3) & (df_salt["x1_err"] < 1)
mask_fit = (df_salt["t0_err"] < 2) & (df_salt["fitprob"] > 1e-4) & (df_salt["mwebv"] < 0.3)

df_salt_selected = df_salt[mask_c & mask_x1 & mask_fit]
snias_new_selected = snias_new[snias_new["sn"].isin(df_salt_selected["sn"])]

In [ ]:
df_salt_selected["mb"] = -2.5 * np.log10(df_salt_selected["x0"]) + 10.502
df_salt_selected["mb_err"] = 2.5 * df_salt_selected["x0_err"] / (np.log(10) * df_salt_selected["x0"])
snias_new_selected["mb"] = -2.5 * np.log10(snias_new_selected["x0"]) + 10.502

In [ ]:
df_salt_selected.sort_values(by="sn", inplace=True)
df_salt_selected.reset_index(inplace=True)
snias_new_selected.sort_values(by="sn", inplace=True)
snias_new_selected.reset_index(inplace=True)

In [ ]:
print("Sucessful SALT fit")
print(len(snias_detected[snias_detected["sn"].isin(df_salt.dropna()["sn"])]))

In [ ]:
print("After SALT cuts")
print(len(snias_new_selected))

In [ ]:
pull_x1 = (df_salt_selected["x1"] - snias_new_selected["x1"]) / df_salt_selected["x1_err"]
pull_c = (df_salt_selected["c"] - snias_new_selected["c"]) / df_salt_selected["c_err"]
pull_t0 = (df_salt_selected["t0"] - snias_new_selected["t0"]) / df_salt_selected["t0_err"]
pull_mb = (df_salt_selected["mb"] - snias_new_selected["mb"]) / df_salt_selected["mb_err"]
pull_x0 = (df_salt_selected["x0"] - snias_new_selected["x0"]) / df_salt_selected["x0_err"]

In [ ]:
nbins = 30

bins_pull = np.linspace(-5, 5, nbins)
xpull = np.linspace(-5, 5, 200)

fig, ax = plt.subplots(4, 3, figsize=(14, 8))

ax[0, 0].hist(snias_new_selected["x1"], bins=np.linspace(-3, 3, nbins), color="teal", alpha=0.3)
ax[0, 0].set_xlabel(r"True $x_1$")
ax[1, 0].hist(pull_x1, bins=bins_pull, color="darkred", alpha=0.5, density=True)
ax[1, 0].plot(xpull, norm.pdf(xpull), color="Black", alpha=0.5, label=r"$\mathcal{N(0,1)}$")
ax[1, 0].set_xlabel(r"Pull on $x_1$")
ax[1, 0].legend()
ax[0, 0].set_ylabel(r"Number of SNe")
ax[1, 0].set_ylabel(r"PDF")


ax[0, 1].hist(snias_new_selected["c"], bins=np.linspace(-0.3, 0.8, nbins), color="teal", alpha=0.3)
ax[0, 1].set_xlabel(r"True $c$")
ax[1, 1].hist(pull_c, bins=bins_pull, color="darkred", alpha=0.5, density=True)
ax[1, 1].plot(xpull, norm.pdf(xpull), color="Black", alpha=0.5, label=r"$\mathcal{N(0,1)}$")
ax[1, 1].set_xlabel(r"Pull on $c$")

ax[0, 2].hist(df_salt_selected["t0"], bins=np.linspace(60823, 61060, nbins), color="teal", alpha=0.3)
ax[0, 2].set_xlabel(r"True $t_0$")
ax[1, 2].hist(pull_t0, bins=bins_pull, color="darkred", alpha=0.5, density=True)
ax[1, 2].plot(xpull, norm.pdf(xpull), color="Black", alpha=0.5, label=r"$\mathcal{N(0,1)}$")
ax[1, 2].set_xlabel(r"Pull on $t_0$")

ax[2, 0].hist(df_salt_selected["mb"], bins=np.linspace(19, 25, nbins), color="teal", alpha=0.3)
ax[2, 0].set_xlabel(r"$m_B$")
ax[3, 0].hist(pull_mb, bins=bins_pull, color="darkred", alpha=0.5, density=True)
ax[3, 0].plot(xpull, norm.pdf(xpull), color="Black", alpha=0.5, label=r"$\mathcal{N(0,1)}$")
ax[3, 0].set_xlabel(r"Pull on $m_B$")
ax[2, 0].set_ylabel(r"Number of SNe")
ax[3, 0].set_ylabel(r"Number of SNe")


for i in range(4):
    for j in range(3):
        ax[i, j].spines["top"].set_visible(False)
        ax[i, j].spines["right"].set_visible(False)
        ax[i, j].spines["left"].set_visible(False)
        ax[i, j].spines["bottom"].set_visible(False)

fig.suptitle("All DDFs, " + str(int(len(pull_x1))) + " SNe using true $z$")

fig.delaxes(ax[3, 2])
fig.delaxes(ax[2, 2])
fig.delaxes(ax[2, 1])
fig.delaxes(ax[3, 1])


plt.tight_layout()
# plt.savefig('/Users/mginolin/Documents/stats_elaiss_DP2.pdf', format='pdf', dpi=1200, bbox_inches='tight')

plt.show()

In [ ]:
# Missing plots here

### Comparison true vs simulated LCs

In [ ]:
i = np.random.randint(0, 144)
i = 10

In [ ]:
sn_name = "LSST_" + str(i)

In [ ]:
sn = true_sne.iloc[i]
params = {"z": sn["z"], "t0": sn["t0"], "x0": sn["x0"], "x1": sn["x1"], "c": sn["c"], "mwebv": sn["mwebv"]}

data_lc = pandas.DataFrame(sn["diaObjectForcedSource_dia_object_lc"])
data_lc.rename(
    columns={"midpointMjdTai": "mjd", "psfDiffFlux": "flux", "psfDiffFluxErr": "fluxerr"}, inplace=True
)

In [ ]:
data_lc

In [ ]:
# saltpars

Look at SALT fit (if it exists)

In [ ]:
sn = snia_data[snia_data["sn"] == sn_name]
lc = lcs[lcs["sn"] == sn_name]
print(sn["sn"].iloc[0])

In [ ]:
# sn_fit = df_salt_selected[df_salt_selected['sn'] == sn['sn'].iloc[0]]

In [ ]:
dust = sncosmo.F99Dust(r_v=3.1)
true_model = sncosmo.Model(source="salt3", effects=[dust], effect_names=["mw"], effect_frames=["obs"])
# Don't forget to add the MW extinction to the SALT model
params = {
    "z": sn["z"].iloc[0],
    "t0": sn["t0"].iloc[0],
    "x0": sn["x0"].iloc[0],
    "x1": sn["x1"].iloc[0],
    "c": sn["c"].iloc[0],
    "mwebv": sn["mwebv"].iloc[0],
}
true_model.update(params)
phase_min = sn["t0"].iloc[0] - 40 * (1 + sn["z"].iloc[0])
phase_max = sn["t0"].iloc[0] + 60 * (1 + sn["z"].iloc[0])

In [ ]:
print(params)

In [ ]:
salt_fit = sncosmo.Model(source="salt3", effects=[dust], effect_names=["mw"], effect_frames=["obs"])
params = {
    "z": sn_fit["z"].iloc[0],
    "t0": sn_fit["t0"].iloc[0],
    "x0": sn_fit["x0"].iloc[0],
    "x1": sn_fit["x1"].iloc[0],
    "c": sn_fit["c"].iloc[0],
    "mwebv": sn["mwebv"].iloc[0],
}
salt_fit.update(params)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 5))

# Converting the phase from rest- to observer-frame
xphase = np.linspace(
    sn["t0"].iloc[0] - 40 * (1 + sn["z"].iloc[0]), sn["t0"].iloc[0] + 60 * (1 + sn["z"].iloc[0]), 1000
)

peaktime = sn["t0"].iloc[0]

bands = np.unique(np.unique(lc["band"]))
c = ["C0", "C1", "C2", "C3", "C4", "C5"]

for i in range(len(bands)):
    band = bands[i]
    lc_band = lc[(lc["band"] == band) & (lc["mjd"].between(phase_min, phase_max))]
    ax.scatter(lc_band["mjd"] - peaktime, lc_band["flux_prime"], s=5, label=band, c=c[i])
    ax.errorbar(
        lc_band["mjd"] - peaktime,
        lc_band["flux_prime"],
        yerr=lc_band["fluxerr_prime"],
        fmt="none",
        zorder=2,
        alpha=0.5,
        c=c[i],
    )
    ax.plot(
        xphase - peaktime, true_model.bandflux(band, xphase, zp=30, zpsys="ab"), ls="dashed", c=c[i], lw=1
    )
    # ax.plot(xphase-peaktime, salt_fit.bandflux(band,  xphase, zp=30, zpsys='ab'), c=c[i], lw=1)

ax.plot([-100, -90], [0, 0], lw=1, c="Black", ls="dashed", label="True SALT3 model")
ax.plot([-100, -90], [0, 0], lw=1, c="Black", label="Fitted SALT3 model")

# ax.text(-40, 375, r'$z^\mathrm{true}=$'+str(round(sn['z'].iloc[0], 3))+r', $x_1^\mathrm{true}=$'+str(round(sn['x1'].iloc[0], 3))+r', $c^\mathrm{true}=$'+str(round(sn['c'].iloc[0], 3)))
# ax.text(-40, 400, r'$z^\mathrm{fit}=$'+str(round(sn_fit['z'].iloc[0], 3))+r'$\pm$'+str(round(sn_fit['z_err'].iloc[0], 3))+r', $x_1^\mathrm{fit}=$'+str(round(sn_fit['x1'].iloc[0], 3))+r'$\pm$'+str(round(sn_fit['x1_err'].iloc[0], 3))+r', $c^\mathrm{fit}=$'+str(round(sn_fit['c'].iloc[0], 3))+r'$\pm$'+str(round(sn_fit['c_err'].iloc[0], 3)))
ax.legend()

# ax.set_yticks([0])
ax.set_ylabel("Flux")
ax.set_xlabel(r"Observer-frame phase (in days)")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.spines["bottom"].set_visible(False)

ax.set_xlim(-40, 60)
# plt.savefig('/Users/mginolin/Documents/example_LC_elaiss_DP2.pdf', format='pdf', dpi=1200, bbox_inches='tight')
plt.show()

In [ ]:
i = 10
sn = true_sne.iloc[i]
saltpars = {"z": sn["z"], "t0": sn["t0"], "x0": sn["x0"], "x1": sn["x1"], "c": sn["c"], "mwebv": sn["mwebv"]}

data_lc = pandas.DataFrame(sn["diaObjectForcedSource_dia_object_lc"]).rename(
    columns={"midpointMjdTai": "mjd", "psfDiffFlux": "flux", "psfDiffFluxErr": "fluxerr"}
)

In [ ]:
sn

In [ ]:
data_lc

In [ ]:
colors = {"u": "pink", "g": "green", "r": "red", "i": "purple", "z": "orange", "y": "brown"}


def plot_lc(lc, saltpars=None):
    maxflux = 0.0
    for f in lc["band"].unique():
        lc_f = lc.loc[lc["band"] == f]
        is_det = np.absolute(lc_f["snr"]) > 5
        lc_det = lc_f.loc[is_det]
        lc_nondet = lc_f.loc[~is_det]
        plt.errorbar(lc_det["mjd"], lc_det["flux"], yerr=lc_det["fluxerr"], fmt="o", label=f, c=colors[f])
        plt.errorbar(
            lc_nondet["mjd"], lc_nondet["flux"], yerr=lc_nondet["fluxerr"], fmt=">", alpha=0.3, c=colors[f]
        )

        if saltpars is not None:
            model = sncosmo.Model(
                "salt3", effects=[sncosmo.F99Dust()], effect_names=["mw"], effect_frames=["obs"]
            )
            model.update(saltpars)
            time = np.linspace(-20, 50, 100) * (1.0 + saltpars["z"]) + saltpars["t0"]
            try:
                flux = model.bandflux("lsst" + f, time, zp=31.4, zpsys="ab")
                maxflux = np.max(np.append(flux, maxflux))
                plt.plot(time, flux, c=colors[f], alpha=0.1, ls="--")
            except:
                continue
        plt.legend()
    is_det = np.absolute(lc["snr"]) > 5
    if saltpars:
        ylim = (-0.2 * maxflux, maxflux * 1.5)
    else:
        ylim = (-0.2 * lc.loc[is_det]["flux"].max(), lc.loc[is_det]["flux"].max() * 1.5)
    # plt.ylim(ylim)


fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
plt.sca(axes[0])
plot_lc(data_lc, saltpars)
plt.xlabel("MJD")
plt.ylabel("Flux (nJy)")
plt.title(f"Real Data (DDF)  z={saltpars['z']:.3f}")
plt.sca(axes[1])
# plot_lc(sim_lc, saltpars)
plt.xlabel("MJD")
plt.title(f"Simulation  z={saltpars['z']:.3f}")
plt.ylim((-3000, 15000))
plt.tight_layout()
plt.show()

In [ ]:
plt.scatter(data_lc_g["mjd"][mask_mjd_data], data_lc_g["flux"][mask_mjd_data])
plt.plot(xphase, true_model.bandflux("rubing", xphase, zp=31.4, zpsys="ab"), ls="dashed", lw=1)

In [ ]:
plt.scatter(sim_lc_g["mjd"][mask_mjd_sim], sim_lc_g["flux"][mask_mjd_sim])
plt.plot(xphase, true_model.bandflux("rubing", xphase, zp=32, zpsys="ab"), ls="dashed", lw=1)

In [ ]:
sim_lc_g

In [ ]:
np.mean(lsst.data[(lsst.data["fieldid"] == 415788) & (lsst.data["band"] == "rubing")]["zp"])

In [ ]:
lsst.data[(lsst.data["fieldid"] == 415788) & (lsst.data["band"] == "rubing")]

In [ ]:
logs_full

In [ ]:
logs_full